In [ ]:
import pandas as pd
import json
import os

def convert_json_to_csv(json_file_path, csv_output_path, nested=False):
    """
    Reads a JSON file and exports it to a CSV.
    
    Args:
        json_file_path (str): Path to the source .json file.
        csv_output_path (str): Path where the .csv will be saved.
        nested (bool): If True, uses json_normalize to flatten deep structures.
    """
    try:
        # Load the JSON data
        with open(json_file_path, 'r') as f:
            data = json.load(f)
        
        # Normalize/Flatten data if it's a nested structure
        if nested:
           df = pd.json_normalize(data)
        else:
            df = pd.DataFrame(data)
            
        # Export to CSV
        df.to_csv(csv_output_path, index=False)
        
        print(f" Success! File saved to: {csv_output_path}")
        return df.head() # Display first 5 rows in the notebook
        
    except Exception as e:
        print(f"Error: {e}")

# %%
# Define paths
input_file = 'afl_video_tracking.json'  # Replace with your filename
output_file = 'data_converted.csv'

# Run conversion
convert_json_to_csv(input_file, output_file, nested=True)

In [1]:
# %% [markdown]
# # AFL Player Tracking: JSON to CSV Flattening
# Converts nested Yolov11 + ByteTrack inference results into a flat tracking CSV.

# %%
import pandas as pd
import json

def process_afl_tracking_json(json_path, csv_output_path):
    """
    Flattens the AFL tracking JSON into the required CSV format.
    """
    try:
        # Load the JSON file
        with open(json_path, 'r') as f:
            data = json.load(f)
        
        # 1. Flatten the 'players' list while keeping frame-level metadata
        df = pd.json_normalize(
            data['tracking_results'], 
            record_path=['players'], 
            meta=['frame_number', 'timestamp'],
            sep='_'
        )
        
        # 2. Rename and Map columns to match your exact expected fields
        column_mapping = {
            'frame_number': 'frame_id',
            'player_id': 'player_id',
            'timestamp': 'timestamps_s',
            'bbox_x1': 'x1',
            'bbox_y1': 'y1',
            'bbox_x2': 'x2',
            'bbox_y2': 'y2',
            'center_x': 'cx',
            'center_y': 'cy',
            'width': 'w',
            'height': 'h',
            'confidence': 'confidence'
        }
        
        # Filter and rename
        df_final = df[list(column_mapping.keys())].rename(columns=column_mapping)
        
        # 3. Export to CSV
        df_final.to_csv(csv_output_path, index=False)
        
        print(f"Successfully converted {len(df_final)} detections to {csv_output_path}")
        return df_final.head()

    except Exception as e:
        print(f" Error during conversion: {e}")

# %%
# Execution
input_json = 'afl_video_tracking.json'
output_csv = 'afl_tracking_results.csv'

process_afl_tracking_json(input_json, output_csv)

Successfully converted 210084 detections to afl_tracking_results.csv


,frame_id,player_id,timestamps_s,x1,y1,x2,y2,cx,cy,w,h,confidence
0,1,1,0.0,637,318,656,355,646,336,19,37,0.79
1,1,2,0.0,948,310,967,358,958,334,19,48,0.79
2,1,3,0.0,920,306,938,355,929,331,18,48,0.77
3,1,4,0.0,860,450,886,513,873,481,25,63,0.76
4,1,5,0.0,6,428,38,489,22,459,32,61,0.76
